# Dióxido de Nitrógeno (NO₂) + Material Particulado Fino (PM2.5 y PM1.0)
### Proyecto GeoVision-CLIP Cali

## 1. Contexto Geo-Industrial

El **Dióxido de Nitrógeno (NO₂)** es uno de los principales contaminantes asociados al **tráfico vehicular** y a la **actividad industrial** en el corredor Cali-Yumbo-Acopi. Junto con las partículas finas **PM2.5** y **PM1.0**, representa uno de los mayores riesgos para la salud respiratoria y cardiovascular de la población.

**Fuentes principales de emisión:**
- Parque Industrial Yumbo-Acopi
- Tráfico pesado en Cañasgordas y Autopista Cali-Yumbo
- Quemas agrícolas estacionales
- Fuentes móviles (diesel)

**Unidades:** Microgramos por metro cúbico (µg/m³)

## 2. Metodología y Fuentes de Datos

### Origen de los datos:

- **2018**: Datos reales proporcionados por DAGMA (estación Universidad del Valle principalmente).
- **2019-2022**: Datos **predichos** debido a la falta de información completa en fuentes abiertas.

### Proceso de obtención y predicción:

1. Se obtuvo un **Application Token** en [datos.gov.co](https://datos.gov.co) para acceder a la API Socrata del IDEAM/DAGMA.
2. Se descargaron todos los registros disponibles (2018-2022) de estaciones en **Santiago de Cali** y **Yumbo**.
3. Se unieron los datos reales de 2018 con los registros gubernamentales.
4. Se entrenó un modelo **XGBoost Regressor** utilizando variables temporales:
   - Hora del día
   - Mes
   - Día de la semana
   - Año
   - Fin de semana

**Precisión del modelo (validación):**
- **MAE**: 5.64 µg/m³
- **RMSE**: 7.85 µg/m³

Los valores de **PM2.5** fueron imputados combinando datos reales del gobierno + predicción del modelo. **PM1.0** se estimó como ≈ 65% de PM2.5 (relación típica observada en zonas urbanas).

**Transparencia**: Todos los archivos generados (`2018DAGMASISNO2.csv` a `2022DAGMASISNO2.csv`) contienen una combinación de datos reales y valores predichos.

In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Cargar los archivos generados
archivos = [f"{year}DAGMASISNO2.csv" for year in range(2018, 2023)]

df_list = []
for archivo in archivos:
    temp = pd.read_csv(archivo)
    temp['Año'] = int(archivo[:4])
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)
df['Fecha'] = pd.to_datetime(df['datetime'])

print(f" Dataset completo cargado: {df.shape[0]:,} registros")
print(f"Período: {df['Fecha'].min().date()} → {df['Fecha'].max().date()}")
print("\nDisponibilidad de datos:")
print(df[['NO2', 'PM2.5', 'PM1.0']].count())

 Dataset completo cargado: 43,824 registros
Período: 2018-01-01 → 2022-12-31

Disponibilidad de datos:
NO2      43824
PM2.5    43824
PM1.0    43824
dtype: int64


## 3. Análisis Exploratorio

In [3]:
df_melt = pd.melt(df, id_vars=['Fecha'],
                  value_vars=['NO2', 'PM2.5', 'PM1.0'],
                  var_name='Contaminante', value_name='Concentración')

fig1 = px.box(df_melt, x='Contaminante', y='Concentración', color='Contaminante',
              title="<b>Distribución General de Contaminantes (2018-2022)</b>",
              template="plotly_dark", points="outliers")

fig1.update_layout(showlegend=False, yaxis_title="Concentración (µg/m³)")
fig1.show()

In [4]:
df_mensual = df.set_index('Fecha').resample('M')[['NO2', 'PM2.5', 'PM1.0']].mean().reset_index()

fig2 = px.line(df_mensual, x='Fecha', y=['NO2', 'PM2.5', 'PM1.0'],
               title="<b>Evolución Mensual Promedio de NO₂, PM2.5 y PM1.0</b>",
               template="plotly_dark", line_shape="spline")

fig2.update_traces(line_width=2.8)
fig2.show()

/tmp/ipykernel_34439/1407902604.py:1: FutureWarning:

'M' is deprecated and will be removed in a future version, please use 'ME' instead.



## 4. Conclusiones y Recomendaciones

- Los niveles de NO₂ muestran patrones claros de mayor contaminación en horas pico y temporada seca.
- La imputación/predicción permitió completar 5 años completos de datos, fundamental para el análisis geoespacial del proyecto.
- Se recomienda complementar este dataset con datos meteorológicos (velocidad y dirección del viento) para mejorar la precisión de futuros modelos.

**Limitación importante**: Parte de los datos (especialmente 2019-2022) son predicciones basadas en patrones temporales. Para estudios de impacto en salud se sugiere validar con mediciones en campo.

In [5]:
df.to_parquet('cali_no2_pm25_pm10_2018_2022.parquet', engine='pyarrow', index=False)
df.to_csv('cali_no2_pm25_pm10_2018_2022.csv', index=False)

print("Archivos guardados correctamente para el proyecto GeoVision.")

Archivos guardados correctamente para el proyecto GeoVision.
